# A/B testing Bayesiano em farmacovigil?ncia: Semaglutida vs Tirzepatida

**Autor:** Luanda Rodrigues | **Papel:** Analista de Dados S?nior  
**Tags:** teste Bayesiano, farmacovigil?ncia, healthcare analytics, openFDA, Python

---

## 1. O problema

Relato de evento adverso n?o ? a mesma coisa que risco cl?nico. Ainda assim, esses relatos s?o um bom ponto de partida quando a pergunta ? operacional: onde vale olhar com mais cuidado?

Neste notebook eu comparo Semaglutida e Tirzepatida usando dados p?blicos do openFDA/FAERS desde janeiro de 2024. A pergunta ? estreita de prop?sito: entre os relatos dispon?veis, qual ? a chance de a propor??o de eventos marcados como graves ser maior para Semaglutida?

Uso um modelo Beta-Binomial porque ele deixa a incerteza vis?vel. Em vez de terminar com um p-valor seco, a an?lise devolve uma probabilidade e um intervalo cred?vel.

> Nota metodol?gica: FAERS/openFDA re?ne notifica??es espont?neas. Serve para sinaliza??o e prioriza??o de investiga??o. N?o mede incid?ncia populacional, n?o controla exposi??o e n?o prova causalidade cl?nica.

In [ ]:
# Configuração de ambiente e bibliotecas
from datetime import date
import warnings

import requests
import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

BASE_URL = 'https://api.fda.gov/drug/event.json'
START_DATE = '20240101'
END_DATE = date.today().strftime('%Y%m%d')
DRUGS = {
    'SEMAGLUTIDE': 'Semaglutida',
    'TIRZEPATIDE': 'Tirzepatida',
}
COLORS = {
    'Semaglutida': '#0f766e',
    'Tirzepatida': '#7c3aed',
    'Accent': '#f59e0b',
}
RNG = np.random.default_rng(42)

print(f'Janela de coleta: {START_DATE} a {END_DATE}')

## 2. Dados

A coleta vem direto da API de eventos adversos da FDA. Busco duas coisas:

1. A s?rie mensal de relatos, usando `receiptdate`.
2. A contagem de relatos graves e n?o graves, usando `serious`.

Tamb?m guardo o `last_updated` retornado pela API. Parece detalhe, mas faz diferen?a quando algu?m abre o notebook meses depois e tenta entender por que os n?meros mudaram.

In [ ]:
def make_search_query(drug_name):
    return f'patient.drug.medicinalproduct:"{drug_name}" AND receiptdate:[{START_DATE} TO {END_DATE}]'


def fda_count(drug_name, count_field):
    params = {
        'search': make_search_query(drug_name),
        'count': count_field,
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    if response.status_code == 404:
        return [], response.json().get('meta', {})
    response.raise_for_status()
    payload = response.json()
    return payload.get('results', []), payload.get('meta', {})


def fetch_fda_timeseries(drug_name):
    results, meta = fda_count(drug_name, 'receiptdate')
    if not results:
        return pd.DataFrame(columns=['time', 'count', 'drug', 'cumulative_count']), meta

    df = pd.DataFrame(results)
    df['time'] = pd.to_datetime(df['time'], format='%Y%m%d')
    df['count'] = pd.to_numeric(df['count'], errors='coerce').fillna(0).astype(int)
    df = (
        df.set_index('time')[['count']]
        .resample('MS')
        .sum()
        .reset_index()
    )
    df['drug'] = DRUGS[drug_name]
    df['cumulative_count'] = df['count'].cumsum()
    return df, meta


frames = []
metadata = {}
for drug in DRUGS:
    df_drug, meta = fetch_fda_timeseries(drug)
    frames.append(df_drug)
    metadata[drug] = meta

df_timeseries = pd.concat(frames, ignore_index=True)
if df_timeseries.empty:
    raise ValueError('A API openFDA nao retornou dados para a janela selecionada.')

df_timeseries['month_str'] = df_timeseries['time'].dt.strftime('%Y-%m')
last_updated = next((m.get('last_updated') for m in metadata.values() if m.get('last_updated')), 'nao informado')
print(f'openFDA last_updated: {last_updated}')
df_timeseries.tail()

## 3. Primeiro olhar: volume de relatos

O gr?fico abaixo mostra relatos acumulados desde janeiro de 2024.

Volume bruto aqui ? barulhento. Ele mistura tempo de mercado, exposi??o, m?dia, comportamento de notifica??o e provavelmente duplicidade. Por isso eu leio esse gr?fico como atividade de notifica??o, n?o como risco absoluto.

In [ ]:
# Evolucao mensal acumulada
max_y = max(1, df_timeseries['cumulative_count'].max()) * 1.1
fig = px.bar(
    df_timeseries,
    x='drug',
    y='cumulative_count',
    color='drug',
    animation_frame='month_str',
    animation_group='drug',
    range_y=[0, max_y],
    color_discrete_map=COLORS,
    title='Relatos acumulados na FDA desde Jan/2024',
    labels={
        'cumulative_count': 'Notificacoes acumuladas',
        'drug': 'Medicamento',
        'month_str': 'Mes',
    },
)

fig.update_layout(
    template='plotly_white',
    title_font_size=20,
    font=dict(family='Arial, sans-serif'),
    showlegend=False,
)

if fig.layout.updatemenus:
    fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 400

fig.show()

## 4. Compara??o Bayesiana da propor??o de relatos graves

A compara??o principal usa a propor??o de relatos marcados como graves entre todos os relatos com `serious` preenchido.

Isso ainda ? observacional. O ganho ? outro: em vez de comparar s? totais, o modelo pergunta como a taxa de gravidade pode variar dentro da incerteza dos dados dispon?veis.

In [ ]:
def fetch_fda_seriousness(drug_name):
    results, meta = fda_count(drug_name, 'serious')
    counts = {item.get('term'): int(item.get('count', 0)) for item in results}
    serious = counts.get(1, 0)
    non_serious = counts.get(2, 0)
    return serious, non_serious, meta


summary_rows = []
for drug, label in DRUGS.items():
    serious, non_serious, meta = fetch_fda_seriousness(drug)
    total = serious + non_serious
    if total == 0:
        raise ValueError(f'Nenhum relato com campo serious foi encontrado para {label}.')
    summary_rows.append({
        'drug_code': drug,
        'Medicamento': label,
        'Graves': serious,
        'Nao graves': non_serious,
        'Total': total,
        'Taxa observada': serious / total,
        'openFDA last_updated': meta.get('last_updated', 'nao informado'),
    })

seriousness_df = pd.DataFrame(summary_rows)
display_df = seriousness_df.copy()
display_df['Taxa observada'] = display_df['Taxa observada'].map(lambda value: f'{value:.1%}')
display_df.drop(columns=['drug_code'])

### 4.1. Modelo Beta-Binomial

Para cada medicamento, trato a taxa real de relatos graves como uma distribui??o. Uso `Beta(1, 1)` como prior inicial e atualizo com os dados observados.

`Posterior = Beta(alpha_prior + graves, beta_prior + total - graves)`

Depois calculo a m?dia posterior e o intervalo cred?vel de 95%. ? uma forma compacta de mostrar o resultado sem fingir precis?o demais.

In [ ]:
alpha_prior = 1
beta_prior = 1
posteriors = {}
posterior_rows = []

for _, row in seriousness_df.iterrows():
    posterior = stats.beta(alpha_prior + row['Graves'], beta_prior + row['Total'] - row['Graves'])
    posteriors[row['Medicamento']] = posterior
    ci_low, ci_high = posterior.ppf([0.025, 0.975])
    posterior_rows.append({
        'Medicamento': row['Medicamento'],
        'Taxa observada': row['Taxa observada'],
        'Media posterior': posterior.mean(),
        'ICr 95% inferior': ci_low,
        'ICr 95% superior': ci_high,
    })

posterior_summary = pd.DataFrame(posterior_rows)
posterior_summary_display = posterior_summary.copy()
for col in ['Taxa observada', 'Media posterior', 'ICr 95% inferior', 'ICr 95% superior']:
    posterior_summary_display[col] = posterior_summary_display[col].map(lambda value: f'{value:.2%}')
posterior_summary_display

### 4.2. Curvas posteriores

As curvas mostram quais taxas de relatos graves continuam plaus?veis depois de observar os dados. Se a sobreposi??o for pequena, a diferen?a entre os grupos fica bem menos amb?gua.

In [ ]:
low = max(0, posterior_summary['ICr 95% inferior'].min() - 0.05)
high = min(1, posterior_summary['ICr 95% superior'].max() + 0.05)
x = np.linspace(low, high, 1000)

fig = go.Figure()
for label, posterior in posteriors.items():
    fig.add_trace(go.Scatter(
        x=x,
        y=posterior.pdf(x),
        mode='lines',
        fill='tozeroy',
        name=label,
        marker_color=COLORS[label],
        opacity=0.72,
    ))

fig.update_layout(
    title='Distribuicao posterior da taxa de relatos graves',
    xaxis_title='Proporcao verdadeira de relatos graves',
    yaxis_title='Densidade de probabilidade',
    template='plotly_white',
    font=dict(family='Arial, sans-serif'),
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

## 5. A pergunta executiva

Agora vem a parte que um dashboard comum n?o responde bem: em quantas simula??es a taxa de relatos graves da Semaglutida fica acima da taxa da Tirzepatida?

In [ ]:
n_simulations = 100_000
sim_sema = posteriors['Semaglutida'].rvs(n_simulations, random_state=RNG)
sim_tirz = posteriors['Tirzepatida'].rvs(n_simulations, random_state=RNG)

diff = sim_sema - sim_tirz
prob_sema_maior = np.mean(diff > 0)
diff_low, diff_high = np.quantile(diff, [0.025, 0.975])

print(
    'Chance estimada de a Semaglutida ter maior proporcao de relatos graves: '
    f'{prob_sema_maior:.4%}'
)
print(f'Diferenca media posterior: {diff.mean():.2%} pontos percentuais')
print(f'Intervalo credivel 95% da diferenca: [{diff_low:.2%}, {diff_high:.2%}]')

## 6. Leitura final

Na janela analisada, os relatos de Semaglutida aparecem com propor??o maior de classifica??o grave do que os de Tirzepatida. O modelo estima essa diferen?a como uma probabilidade direta e tamb?m como diferen?a m?dia entre as duas taxas.

Isso ? ?til para farmacovigil?ncia porque aponta onde revisar eventos com mais aten??o. N?o ? um ranking cl?nico de seguran?a. Para isso, faltam exposi??o, denominador de pacientes, controle de confundidores e desenho causal.

Para uma opera??o de sa?de, eu usaria esse notebook como triagem. Repetiria a an?lise quando a openFDA atualizasse a base, acompanharia a s?rie mensal e levaria os achados para uma revis?o cl?nica mais cuidadosa antes de mexer em protocolo.